## Лабораторна робота №2 — Порівняння методів класифікації даних

**Мета:** за допомогою Python (scikit-learn) дослідити різні методи класифікації та навчитися їх порівнювати.

**Джерела даних:**
- `income_data.txt` (Adult/Census Income)
- `Iris` (вбудований у `sklearn.datasets` та CSV-варіант)

> У методичці є фрагменти коду з помилками (імена змінних, імпорти, індексація тощо). У цьому ноутбуці все доведено до робочого стану.


In [ ]:
# Базові імпорти
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.svm import LinearSVC, SVC
from sklearn.multiclass import OneVsOneClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

import matplotlib.pyplot as plt

LAB_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
LAB2_DIR = LAB_DIR
print('Поточна директорія:', LAB2_DIR)
print('Файли:', os.listdir(LAB2_DIR)[:10])


## Завдання 2.1. Класифікація за допомогою SVM (LinearSVC)

### 14 ознак (атрибутів) датасету Census Income (Adult)

| № | Назва ознаки | Тип | Що означає |
|---:|---|---|---|
| 1 | `age` | числова | вік (роки) |
| 2 | `workclass` | категоріальна | тип зайнятості/роботодавця (Private, State-gov, …) |
| 3 | `fnlwgt` | числова | фінальна вага запису в вибірці (ваговий коефіцієнт перепису) |
| 4 | `education` | категоріальна | рівень освіти (HS-grad, Bachelors, …) |
| 5 | `education-num` | числова | освіта у вигляді числа (кількість років/ранг) |
| 6 | `marital-status` | категоріальна | сімейний стан |
| 7 | `occupation` | категоріальна | професія/вид робіт |
| 8 | `relationship` | категоріальна | роль у сім’ї (Husband, Wife, …) |
| 9 | `race` | категоріальна | раса |
| 10 | `sex` | категоріальна | стать |
| 11 | `capital-gain` | числова | дохід від капіталу |
| 12 | `capital-loss` | числова | втрати капіталу |
| 13 | `hours-per-week` | числова | годин на тиждень |
| 14 | `native-country` | категоріальна | країна походження |

**Цільова змінна (мітка класу):** `income` ∈ {`<=50K`, `>50K`}.

> Далі будуємо бінарний класифікатор, який прогнозує, чи перевищує дохід $50k.


In [ ]:
# Завантаження та підготовка income_data.txt
columns = [
    'age','workclass','fnlwgt','education','education-num','marital-status','occupation',
    'relationship','race','sex','capital-gain','capital-loss','hours-per-week','native-country','income'
]

# Знайдемо файл незалежно від поточного cwd
candidate_paths = [
    os.path.join(os.getcwd(), 'income_data.txt'),
    os.path.join(os.getcwd(), 'Lab2', 'income_data.txt'),
    os.path.join(LAB2_DIR, 'income_data.txt'),
]
input_path = next((p for p in candidate_paths if os.path.exists(p)), None)
input_path


In [ ]:
df = pd.read_csv(
    input_path,
    header=None,
    names=columns,
    sep=r',\s*',
    engine='python',
    na_values='?',
)

print('Розмірність (рядки, колонки):', df.shape)
print('Приклад рядків:')
df.head()


In [ ]:
# Приберемо рядки з пропусками, як у методичці (рядки з '?')
df_clean = df.dropna().copy()

print('Після dropna:', df_clean.shape)
print('Розподіл класів:')
print(df_clean['income'].value_counts())


In [ ]:
# Візьмемо до 25_000 записів для кожного класу (як в методичці)
max_per_class = 25_000

df_le = df_clean[df_clean['income'] == '<=50K'].head(max_per_class)
df_gt = df_clean[df_clean['income'] == '>50K'].head(max_per_class)

df_balanced = pd.concat([df_le, df_gt], axis=0).sample(frac=1.0, random_state=42).reset_index(drop=True)
print('Збалансована вибірка:', df_balanced['income'].value_counts().to_dict())
print('Розмірність:', df_balanced.shape)


In [ ]:
# Розбиття 80/20
X_df = df_balanced.drop(columns=['income'])
y = df_balanced['income']

X_train, X_test, y_train, y_test = train_test_split(
    X_df, y,
    test_size=0.2,
    random_state=5,
    stratify=y,
)

X_train.shape, X_test.shape


In [ ]:
# Побудова препроцесингу: числові як є (з масштабуванням), категоріальні — OneHot
numeric_features = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
categorical_features = [c for c in X_df.columns if c not in numeric_features]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler(with_mean=False)),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

# Лінійний SVM
svm_linear = LinearSVC(random_state=0)

model_21 = Pipeline(steps=[
    ('preprocess', preprocess),
    ('clf', svm_linear),
])

model_21


In [ ]:
# Навчання
model_21.fit(X_train, y_train)

# Прогноз на тесті
y_pred = model_21.predict(X_test)

metrics_21 = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision(>50K)': precision_score(y_test, y_pred, pos_label='>50K'),
    'recall(>50K)': recall_score(y_test, y_pred, pos_label='>50K'),
    'f1(>50K)': f1_score(y_test, y_pred, pos_label='>50K'),
    'f1_weighted': f1_score(y_test, y_pred, average='weighted'),
}
metrics_21


In [ ]:
print('Classification report:')
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=['<=50K','>50K'])
cm


In [ ]:
# Перехресна перевірка (як аналог з методички): F1-weighted, 3-fold
cv_f1 = cross_val_score(model_21, X_df, y, scoring='f1_weighted', cv=3)
print('F1_weighted (CV=3):', cv_f1.mean(), '+/-', cv_f1.std())


In [ ]:
# Прогноз для тестової точки з методички
input_data = {
    'age': 37,
    'workclass': 'Private',
    'fnlwgt': 215646,
    'education': 'HS-grad',
    'education-num': 9,
    'marital-status': 'Never-married',
    'occupation': 'Handlers-cleaners',
    'relationship': 'Not-in-family',
    'race': 'White',
    'sex': 'Male',
    'capital-gain': 0,
    'capital-loss': 0,
    'hours-per-week': 40,
    'native-country': 'United-States',
}
input_df = pd.DataFrame([input_data])
pred = model_21.predict(input_df)[0]
print('Передбачений клас доходу:', pred)


### Пояснення до завдання 2.1 (текст для звіту “від себе”)

1. **Що я робив у цьому завданні.** Я розв’язував задачу **бінарної класифікації**: за 14 ознаками людини потрібно визначити, чи її річний дохід **`>50K`** або **`<=50K`**.

2. **Чому не можна одразу подати дані в модель.** У наборі даних є два типи ознак:
   - **числові** (вік, fnlwgt, education-num, capital-gain, capital-loss, hours-per-week) — їх можна використовувати напряму;
   - **категоріальні** (workclass, education, marital-status, occupation, relationship, race, sex, native-country) — це рядки, а SVM не “розуміє” рядки.
   Тому перед навчанням я виконав попередню обробку.

3. **Як я обробив категоріальні ознаки.** Я застосував **one-hot кодування** (`OneHotEncoder`). Це зручно і правильно для категорій, бо не задає штучної “ієрархії” між значеннями (як це було б при простому кодуванні числами).

4. **Як я обробив пропуски.** У файлі трапляються значення `?`. Я трактував їх як пропуски та **видалив** такі рядки (аналогічно до рекомендацій у методичці).

5. **Як я формував вибірку.** Для кожного класу я взяв **до 25 000** записів, потім перемішав дані та зробив розбиття **80/20** на навчальну й тестову частини зі **стратифікацією**, щоб частки класів були однаковими в обох вибірках.

6. **Яку модель я використав.** Я побудував **лінійний SVM** (`LinearSVC`). Суть моделі — знайти лінійну межу (гіперплощину), яка найкраще відділяє два класи в просторі ознак.

7. **Як я оцінював якість.** Я порахував стандартні метрики:
   - **Accuracy** — загальна частка правильних відповідей;
   - **Precision/Recall/F1** для класу `>50K` — щоб окремо оцінити якість знаходження людей з високим доходом;
   - також сформував `classification_report` і **матрицю плутанини**.
   Додатково я зробив **крос-валідацію** для `F1_weighted`.

8. **Висновок про тестову точку.** Для заданого в методичці прикладу (14 ознак) я подав дані в підготовлену модель і отримав прогноз класу (`<=50K` або `>50K`). Цей прогноз я і вважаю відповіддю, до якого класу належить тестова точка.


## Завдання 2.2. Порівняння SVM з нелінійними ядрами

Побудуємо **Kernel SVM** на тих самих даних `income_data.txt` і порівняємо 3 ядра:
- поліноміальне: `kernel='poly'` (у методичці запропоновано `degree=8`)
- гаусове (RBF): `kernel='rbf'`
- сигмоїдальне: `kernel='sigmoid'`

Для кожного варіанта порахуємо **accuracy/precision/recall/F1** (та за потреби додатково подивимось `classification_report`).


In [ ]:
def evaluate_binary(y_true, y_pred, positive='>50K'):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        f'precision({positive})': precision_score(y_true, y_pred, pos_label=positive),
        f'recall({positive})': recall_score(y_true, y_pred, pos_label=positive),
        f'f1({positive})': f1_score(y_true, y_pred, pos_label=positive),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted'),
    }

# ВАЖЛИВО: Kernel SVM (SVC з rbf/poly/sigmoid) має високу обчислювальну складність (приблизно O(n^2)),
# тому на 50 000 прикладів він може працювати дуже довго. Для коректного експерименту беремо меншу підвибірку.
max_per_class_kernel = 2000

df_le_k = df_clean[df_clean['income'] == '<=50K'].head(max_per_class_kernel)
df_gt_k = df_clean[df_clean['income'] == '>50K'].head(max_per_class_kernel)
df_kernel = pd.concat([df_le_k, df_gt_k], axis=0).sample(frac=1.0, random_state=42).reset_index(drop=True)

Xk = df_kernel.drop(columns=['income'])
yk = df_kernel['income']

Xk_train, Xk_test, yk_train, yk_test = train_test_split(
    Xk, yk,
    test_size=0.2,
    random_state=5,
    stratify=yk,
)

kernels = {
    'poly(degree=8)': SVC(kernel='poly', degree=8),
    'rbf': SVC(kernel='rbf'),
    'sigmoid': SVC(kernel='sigmoid'),
}

results_22 = {}
for name, clf in kernels.items():
    model = Pipeline(steps=[
        ('preprocess', preprocess),
        ('clf', clf),
    ])
    model.fit(Xk_train, yk_train)
    pred = model.predict(Xk_test)
    results_22[name] = evaluate_binary(yk_test, pred)

pd.DataFrame(results_22).T


### Пояснення до завдання 2.2 (текст для звіту “від себе”)

1. **Що я хотів перевірити.** У 2.1 я використав лінійний SVM, але він будує лише *лінійну* межу. У цьому пункті я перевірив, як зміниться якість, якщо використати **SVM з нелінійними ядрами (Kernel SVM)**.

2. **Ідея Kernel SVM простими словами.** Ядро дозволяє моделі будувати **нелінійну** межу між класами: умовно кажучи, дані “перетворюються” так, щоб їх стало легше розділяти.

3. **Які ядра я протестував.** Я навчив 3 моделі `SVC`:
   - **поліноміальне ядро** `kernel='poly'` (я взяв `degree=8`, як у методичці);
   - **гаусове (RBF) ядро** `kernel='rbf'`;
   - **сигмоїдальне ядро** `kernel='sigmoid'`.

4. **Як я робив порівняння чесно.** Щоб результати були порівнювані, я залишив:
   - **той самий препроцесинг** (числові ознаки + one-hot для категоріальних);
   - **однакове розбиття 80/20**.

5. **Про обчислювальну складність.** Kernel SVM на великих вибірках може працювати дуже довго, тому для цього експерименту я використав **меншу підвибірку** (щоб моделі реально встигали навчитися і дати результат).

6. **Як я оцінював якість.** Для кожного ядра я порахував **accuracy, precision, recall, F1** (особливо дивився на F1 для класу `>50K`, бо він показує баланс між precision і recall).

7. **Висновок.** Найкращим я вважаю те ядро, яке дало **найвищий F1 (або F1_weighted)** при нормальному співвідношенні precision/recall, тобто модель не просто “вгадує” більшість, а адекватно знаходить клас `>50K`.


## Завдання 2.3. Порівняння якості класифікаторів на прикладі Iris


In [ ]:
# КРОК 1. Завантаження та вивчення Iris (через sklearn.datasets)
from sklearn.datasets import load_iris

iris_dataset = load_iris()
print('Ключі iris_dataset:\n', iris_dataset.keys())
print('\nDESCR (початок):\n', iris_dataset['DESCR'][:250], '\n...')
print('\nНазви відповідей (класи):', iris_dataset['target_names'])
print('Назви ознак:', iris_dataset['feature_names'])

print('\nТип data:', type(iris_dataset['data']))
print('Форма data:', iris_dataset['data'].shape)
print('\nПерші 5 прикладів (X):\n', iris_dataset['data'][:5])

print('\nТип target:', type(iris_dataset['target']))
print('Перші 25 міток target:\n', iris_dataset['target'][:25])


In [ ]:
# Альтернативно: завантаження Iris з URL (як у методичці)
from pandas import read_csv
from pandas.plotting import scatter_matrix

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/iris.csv"
names = ['sepal-length', 'sepal-width', 'petal-length', 'petal-width', 'class']
dataset = read_csv(url, names=names)

print('dataset.shape =', dataset.shape)
print('\nПерші 20 рядків:')
print(dataset.head(20))

print('\nОписова статистика (describe):')
print(dataset.describe())

print('\nРозподіл за класами:')
print(dataset.groupby('class').size())


In [ ]:
# КРОК 2. Візуалізація даних

# Діаграма розмаху (box plot)
dataset.plot(kind='box', subplots=True, layout=(2,2), sharex=False, sharey=False, figsize=(10,6))
plt.suptitle('Boxplots for Iris features')
plt.show()

# Гістограми
dataset.hist(figsize=(10,6))
plt.suptitle('Histograms for Iris features')
plt.show()

# Матриця діаграм розсіювання
scatter_matrix(dataset, figsize=(10,10))
plt.suptitle('Scatter Matrix for Iris dataset')
plt.show()


In [ ]:
# КРОК 3. Розділення на навчальну та контрольну вибірки (80/20)
array = dataset.values
X = array[:, 0:4].astype(float)
y = array[:, 4]

X_train, X_validation, y_train, y_validation = train_test_split(
    X, y,
    test_size=0.20,
    random_state=1,
    stratify=y,
)

X_train.shape, X_validation.shape


In [ ]:
# КРОК 4. Порівняння 6 алгоритмів зі Stratified 10-fold CV (метрика: accuracy)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB

models = []
models.append(('LR', LogisticRegression(solver='liblinear', multi_class='ovr')))
models.append(('LDA', LinearDiscriminantAnalysis()))
models.append(('KNN', KNeighborsClassifier()))
models.append(('CART', DecisionTreeClassifier()))
models.append(('NB', GaussianNB()))
models.append(('SVM', SVC(gamma='auto')))

results = []
names_ = []

for name, model in models:
    kfold = StratifiedKFold(n_splits=10, random_state=1, shuffle=True)
    cv_results = cross_val_score(model, X_train, y_train, cv=kfold, scoring='accuracy')
    results.append(cv_results)
    names_.append(name)
    print(f"{name}: {cv_results.mean():.4f} ({cv_results.std():.4f})")

plt.figure(figsize=(8,4))
plt.boxplot(results, labels=names_)
plt.title('Algorithm Comparison (Iris)')
plt.ylabel('Accuracy')
plt.show()


In [ ]:
# КРОК 6-7. Навчання обраної моделі (SVM) та оцінка на контрольній вибірці
model = SVC(gamma='auto')
model.fit(X_train, y_train)

predictions = model.predict(X_validation)

print('Accuracy on validation:', accuracy_score(y_validation, predictions))
print('\nConfusion matrix:\n', confusion_matrix(y_validation, predictions))
print('\nClassification report:\n', classification_report(y_validation, predictions))


In [ ]:
# КРОК 8. Прогноз для нової квітки
X_new = np.array([[5.0, 2.9, 1.0, 0.2]])
print('Форма масиву X_new:', X_new.shape)

prediction = model.predict(X_new)
print('Прогноз (клас як текст):', prediction[0])


### Пояснення до завдання 2.3 (текст для звіту “від себе”)

1. **Про що це завдання.** Тут я працював з класичним датасетом **Iris**. Моє завдання — за 4 вимірюваннями квітки (довжина/ширина чашолистка і пелюстки) передбачити один із 3 сортів: **setosa / versicolor / virginica**.

2. **Чому це “навчання з учителем”.** У кожному рядку даних уже є правильний сорт (мітка класу), тому я навчаю модель на прикладах “ознаки → правильна відповідь”.

3. **Що я перевірив у структурі даних.** Я подивився:
   - які ключі є в `iris_dataset` (`data`, `target`, `feature_names`, `target_names`, `DESCR`);
   - форму `data` (має бути 150 прикладів і 4 ознаки);
   - приклади перших рядків, щоб зрозуміти масштаби чисел.

4. **Описова статистика та класи.** Через `describe()` я оцінив середні значення і діапазони, а також перевірив, що класи збалансовані (по 50 екземплярів кожного сорту).

5. **Навіщо я будував графіки.**
   - Boxplot показує розкид значень і можливі викиди.
   - Гістограми показують розподіл кожної ознаки.
   - Scatter matrix дає змогу побачити, чи відділяються сорти за парами ознак (і зазвичай Iris добре відділяється, особливо setosa).

6. **Розбиття на train/test.** Я розділив дані у пропорції **80/20** (train/validation), щоб чесно перевірити якість на даних, яких модель не бачила під час навчання.

7. **Порівняння алгоритмів.** Я протестував 6 моделей (LR, LDA, KNN, CART, NB, SVM) і порівняв їх через **Stratified 10-fold cross-validation** за метрикою **accuracy**. Так оцінка більш надійна для невеликого датасету.

8. **Як я обирав “кращий” метод.** Я дивився на **середню accuracy** і на **стабільність** (стандартне відхилення). Модель з більшою середньою точністю та меншою варіативністю я вважаю кращою.

9. **Фінальна перевірка на контрольній вибірці.** Далі я навчив вибрану модель на train і оцінив на validation: accuracy, матриця плутанини, `classification_report`.

10. **Прогноз для нової квітки.** Для `X_new = [5.0, 2.9, 1.0, 0.2]` я отримав прогноз сорту — це і є відповідь, до якого класу модель віднесла новий екземпляр.


## Завдання 2.4. Порівняння якості класифікаторів для `income_data.txt`

По аналогії з 2.3 порівняємо 6 алгоритмів на датасеті з 2.1:
- LR, LDA, KNN, CART, NB, SVM.

Порівняння зробимо через стратифіковану крос-валідацію (accuracy), а також додатково подивимось метрики на hold-out тесті.


In [ ]:
# Препроцесинг для порівняння моделей (dense one-hot для сумісності з LDA/KNN/NB)
try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    # для старіших версій sklearn
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

numeric_transformer_dense = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer_dense = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', ohe),
])

preprocess_dense = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_dense, numeric_features),
        ('cat', categorical_transformer_dense, categorical_features),
    ],
    remainder='drop'
)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB

models_income = []
models_income.append(('LR', LogisticRegression(solver='liblinear', max_iter=200)))
models_income.append(('LDA', LinearDiscriminantAnalysis()))
models_income.append(('KNN', KNeighborsClassifier(n_neighbors=15)))
models_income.append(('CART', DecisionTreeClassifier(random_state=1)))
models_income.append(('NB', GaussianNB()))
models_income.append(('SVM', SVC(gamma='scale')))

cv_results_income = []
names_income = []

kfold_income = StratifiedKFold(n_splits=5, random_state=1, shuffle=True)

for name, clf in models_income:
    pipe = Pipeline(steps=[('preprocess', preprocess_dense), ('clf', clf)])
    scores = cross_val_score(pipe, X_df, y, cv=kfold_income, scoring='accuracy')
    cv_results_income.append(scores)
    names_income.append(name)
    print(f"{name}: {scores.mean():.4f} ({scores.std():.4f})")

plt.figure(figsize=(8,4))
plt.boxplot(cv_results_income, labels=names_income)
plt.title('Algorithm Comparison (income_data.txt)')
plt.ylabel('Accuracy (CV)')
plt.show()


In [ ]:
# Додаткова оцінка на тестовій частині (80/20) для кожного алгоритму
holdout_metrics_24 = {}
for name, clf in models_income:
    pipe = Pipeline(steps=[('preprocess', preprocess_dense), ('clf', clf)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    holdout_metrics_24[name] = evaluate_binary(y_test, pred)

pd.DataFrame(holdout_metrics_24).T.sort_values('f1(>50K)', ascending=False)


### Пояснення до завдання 2.4 (текст для звіту “від себе”)

1. **Що я робив.** У цьому пункті я зробив **порівняння кількох алгоритмів** на датасеті `income_data.txt` (такому ж, як у 2.1), щоб визначити, який підхід дає кращу якість класифікації.

2. **Чому потрібен однаковий препроцесинг для всіх моделей.** Дані містять і числа, і категорії. Щоб порівняння було чесним, я зробив один спільний пайплайн підготовки:
   - числові ознаки: заповнення медіаною + масштабування (`StandardScaler`);
   - категоріальні: заповнення найчастішим значенням + **one-hot**.
   Це реалізовано через `ColumnTransformer`, тому кожна модель отримує дані в однаковому форматі.

3. **Як я оцінював моделі.** Основне порівняння я робив через **Stratified K-Fold cross-validation** (я взяв 5 фолдів, щоб обчислення на великому датасеті не були надто довгими). Стратифікація важлива, бо зберігає пропорції класів у кожному фолді.

4. **Які моделі я порівнював.** Я протестував 6 алгоритмів: **LR, LDA, KNN, CART, NB, SVM**.

5. **За якими критеріями я вибирав найкращу модель.**
   - у крос-валідації я дивився на **accuracy** (як у методичці) і на стабільність результату;
   - додатково на hold-out тесті я порахував **precision/recall/F1** для класу `>50K`, щоб краще зрозуміти, як модель знаходить “високий дохід”, а не лише загальну точність.

6. **Висновок.** Після порівняння я обрав алгоритм, який показав найкраще співвідношення метрик (передусім F1 для `>50K`) і достатню стабільність. У звіті я вказую, яка модель вийшла найкращою за отриманими результатами і чому саме її доцільно використовувати.


## Завдання 2.5. Класифікація Iris лінійним класифікатором Ridge

Потрібно виправити код (у методичці є помилки) і виконати класифікацію. Також потрібно пояснити налаштування `RidgeClassifier`, метрики якості, матрицю плутанини та коефіцієнти Коена-Каппа і Метьюза.


In [1]:
# Виправлений код для RidgeClassifier + матриця плутанини
from sklearn.linear_model import RidgeClassifier
from sklearn import metrics

# (якщо seaborn не встановлено, розкоментуйте наступний рядок)
# !pip -q install seaborn

import seaborn as sns
sns.set()

iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=0,
    stratify=y,
)

clf = RidgeClassifier(tol=1e-2, solver='sag')
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print('Accuracy:', np.round(metrics.accuracy_score(y_test, y_pred), 4))
print('Precision (weighted):', np.round(metrics.precision_score(y_test, y_pred, average='weighted'), 4))
print('Recall (weighted):', np.round(metrics.recall_score(y_test, y_pred, average='weighted'), 4))
print('F1 Score (weighted):', np.round(metrics.f1_score(y_test, y_pred, average='weighted'), 4))
print('Cohen Kappa Score:', np.round(metrics.cohen_kappa_score(y_test, y_pred), 4))
print('Matthews Corrcoef:', np.round(metrics.matthews_corrcoef(y_test, y_pred), 4))

print('\nClassification Report:\n', metrics.classification_report(y_test, y_pred, target_names=iris.target_names))

mat = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(mat, square=True, annot=True, fmt='d', cbar=False,
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
plt.xlabel('predicted label')
plt.ylabel('true label')
plt.title('Confusion Matrix (RidgeClassifier on Iris)')

out_img = os.path.join(os.getcwd(), 'Confusion.jpg')
plt.tight_layout()
plt.savefig(out_img, dpi=200)
plt.show()

out_img


NameError: name 'load_iris' is not defined

### Пояснення до завдання 2.5 (текст для звіту “від себе”)

1. **Що я зробив у цьому завданні.** Я взяв приклад з методички для **`RidgeClassifier`**, виправив помилки в коді (імена змінних, імпорти, назви `X_test`/`Xtest` тощо) і запустив класифікацію для датасету Iris.

2. **Що таке `RidgeClassifier`.** Це **лінійний** класифікатор з **L2-регуляризацією (ridge)**. Сенс регуляризації в тому, що вона “штрафує” великі ваги моделі, завдяки чому зменшується перенавчання і модель стає більш стабільною.

3. **Які налаштування я використав і що вони значать.**
   - **`tol = 1e-2`** — критерій зупинки оптимізації: якщо подальше покращення дуже мале, навчання зупиняється.
   - **`solver = 'sag'`** — метод оптимізації *Stochastic Average Gradient*, який добре підходить для задач з регуляризацією.

4. **Метрики якості, які я порахував.**
   - **Accuracy** — яка частка прикладів класифікована правильно.
   - **Precision (weighted)** — зважена точність по класах (вага залежить від кількості прикладів у класі).
   - **Recall (weighted)** — зважена повнота по класах.
   - **F1 (weighted)** — зважене узгодження precision і recall.
   - **Cohen’s Kappa** — показує, наскільки прогноз моделі узгоджується з правильними мітками **понад випадкову згоду** (0 — як випадково, ближче до 1 — дуже добре).
   - **MCC (Matthews Corrcoef)** — кореляційна міра якості класифікації, яка враховує всю матрицю плутанини. Для багатокласової задачі це узагальнений показник: 1 — ідеально, 0 — випадково.

5. **Матриця плутанини та `Confusion.jpg`.** Я побудував матрицю плутанини і зберіг її як `Confusion.jpg`:
   - по осі **y** — істинні класи (true label),
   - по осі **x** — передбачені класи (predicted label),
   - числа на діагоналі — правильні передбачення,
   - позадіагональні — помилки (які класи модель плутає).

6. **Що я заніс у звіт.** У звіт я вставляю значення метрик, `classification_report` і картинку `Confusion.jpg`, а також коротко описую, де саме модель помиляється (за позадіагональними елементами).
